# Nurse Line Cost-Savings Analysis

Matches members across a nurse-call system (SQL Server) and a medical-claims system (MySQL), then estimates how much the nurse line saved by steering members toward less intensive treatment.

Outputs are cleared because the data lives on course databases (synthetic member records). Results are summarized in the README.

# Setup

In [ ]:
# pip install pandas sqlalchemy pyodbc pymysql pandasql

## Imports

In [ ]:
import re
import pandas as pd
from sqlalchemy import create_engine, text
from getpass import getpass
from urllib.parse import quote_plus
import os

## Connect to CallDB (SQL Server)

In [ ]:
server=os.environ['CALLDB_SERVER']        # SQL Server host for the nurse-call database
database='CallDB'
connection_string=f'mssql+pyodbc://@{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server'
callEngine=create_engine(connection_string)

## Connect to MedClaims (MySQL)

In [ ]:
mySQLserver=os.environ['CLAIMS_SERVER']    # MySQL host for the claims database
mySQLDB='MedClaims'
username=os.environ['CLAIMS_USER']
password=getpass('Claims DB password: ')

## Build the MySQL engine

In [ ]:
claimsEngine = create_engine(
    f"mysql+pymysql://{username}:{quote_plus(password)}@{mySQLserver}:3306/{mySQLDB}"
)

## Load members from both platforms

Client 103 from the call database and client 3 from the claims database.

In [ ]:
callmembers = pd.read_sql(text("select * from dbo.members where client_id=103;"), callEngine)
claimMembers = pd.read_sql(text("select * from members where client_id=3;"), claimsEngine)
print(callmembers.shape, claimMembers.shape)

## Standardize names, dates and contact details

The call data is messy nicknames, middle initials, name suffixes and missing phone numbers so both tables are cleaned into the same format before matching.

In [ ]:
def clean_first(s):
    if pd.isna(s): return None
    tokens = re.sub(r'[^a-z ]', ' ', str(s).lower()).split()
    return tokens[0] if tokens else None

def clean_last(s):
    if pd.isna(s): return None
    s = re.sub(r'[^a-z ]', ' ', str(s).lower())
    s = re.sub(r'\b(jr|sr|ii|iii|iv)\b', ' ', s)
    s = ''.join(s.split())
    return s or None

def clean_addr(s):
    if pd.isna(s): return None
    s = re.sub(r'[^a-z0-9]', '', str(s).lower())
    return s or None

def standardize(df, id_col):
    out = pd.DataFrame({'id': df[id_col]})
    out['first']  = df['FIRST_NAME'].apply(clean_first)
    out['f_init'] = out['first'].str[:1]
    out['last']   = df['LAST_NAME'].apply(clean_last)
    out['dob']    = pd.to_datetime(df['DOB'], errors='coerce').dt.date
    out['zip']    = df['ZIP'].astype(str).str.extract(r'(\d+)')[0].str.zfill(5).str[:5]
    ph = df['PHONE'].astype(str).str.replace(r'\D', '', regex=True).str[-10:]
    out['phone']  = ph.where(ph.str.len() == 10)
    out['addr']   = df['ADDRESS'].apply(clean_addr)
    return out

## Apply the standardization

In [ ]:
claims = standardize(claimMembers, 'MEMBER_ID')
calls  = standardize(callmembers, 'CALL_MEMBER_ID')

## Match members across platforms

Matching runs in passes, strictest rule first. Each pass works only on records still unmatched, and ambiguous keys are skipped so every match stays one-to-one.

In [ ]:
passes = [
    ['last', 'first', 'dob'],
    ['last', 'f_init', 'dob'],
    ['dob', 'zip', 'f_init'],
    ['phone'],
    ['addr'],
]

matched = []
c_left, k_left = claims.copy(), calls.copy()

for keys in passes:
    a = c_left.dropna(subset=keys)
    b = k_left.dropna(subset=keys)
    a = a[~a.duplicated(keys, keep=False)]
    b = b[~b.duplicated(keys, keep=False)]
    m = a.merge(b, on=keys, suffixes=('_claim', '_call'))
    matched.append(m[['id_claim', 'id_call']])
    c_left = c_left[~c_left['id'].isin(m['id_claim'])]
    k_left = k_left[~k_left['id'].isin(m['id_call'])]

crosswalk = pd.concat(matched, ignore_index=True).rename(
    columns={'id_claim': 'MEMBER_ID', 'id_call': 'CALL_MEMBER_ID'})
print(crosswalk.shape)

## Check the claims table structures

In [ ]:
print(pd.read_sql(text("describe inpatient_claims;"), claimsEngine)['Field'].tolist())
print(pd.read_sql(text("describe outpatient_claims;"), claimsEngine)['Field'].tolist())

## Pull the nurse call data

In [ ]:
callAnalysisQuery="""
select
    members.CALL_MEMBER_ID,
    call_date,
    pre_intent,
    post_intent,
    condition
from
    dbo.nurse_calls inner join
    dbo.members on nurse_calls.CALL_MEMBER_ID=members.CALL_MEMBER_ID
where
    members.CLIENT_ID=103;
"""

callAnalysisDF=pd.read_sql(text(callAnalysisQuery),callEngine)
print(callAnalysisDF.shape)
callAnalysisDF.head()

## Pull inpatient claims

ICD-9 descriptions are grouped into a condition and HCPCS descriptions into a treatment category.

In [ ]:
claimsAnalysisQuery="""
select
    members.member_id,
    ic.discharge_dt as treatment_date,
    ic.clm_pmt_amt as claim_amount,
    case when icd9_dgns_desc_1 like '%shoulder%' then 'SHOULDER'
        when icd9_dgns_desc_1 like '%rotator cuff%' then 'SHOULDER'
        when icd9_dgns_desc_1 like '%knee%' then 'KNEE'
        when icd9_dgns_desc_1 like '%lower leg%' then 'KNEE'
        when icd9_dgns_desc_1 like '%pelvic region and thigh%' then 'HIP'
        when icd9_dgns_desc_1 like '%hip%' then 'HIP'
        when icd9_dgns_desc_1 like '%back%' then 'BACK'
        when icd9_dgns_desc_1 like '%lumbar%' then 'BACK'
        when icd9_dgns_desc_1 like '%lumbago%' then 'BACK'
        when icd9_dgns_desc_1 like '%intervertebral disc%' then 'BACK'
        else 'OTHER'
    end as CNDTN,
    case when hcpcs_desc_1 like '%arthroplasty%' then 'SURGERY'
        when hcpcs_desc_1 like '%arthrodesis%' then 'SURGERY'
        when hcpcs_desc_1 like '%laminotomy%' then 'SURGERY'
        when hcpcs_desc_1 like '%surgery%' then 'SURGERY'
        when hcpcs_desc_1 like '%therapeutic exercise%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%neuromuscular reeducation%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%manual therapy%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%therapy%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%injection%' then 'MEDICATION'
        when hcpcs_desc_1 like '%arthrocentesis%' then 'MEDICATION'
        when hcpcs_desc_1 like '%vaccine%' then 'MEDICATION'
        when hcpcs_desc_1 like '%radiologic%' then 'IMAGING'
        when hcpcs_desc_1 like '%mri%' then 'IMAGING'
        when hcpcs_desc_1 like '%office or other outpatient visit%' then 'OFFICE VISIT'
        when hcpcs_desc_1 like '%preventive medicine%' then 'OFFICE VISIT'
        when hcpcs_desc_1 like '%electrocardiogram%' then 'LAB/DIAGNOSTIC'
        when hcpcs_desc_1 like '%metabolic panel%' then 'LAB/DIAGNOSTIC'
        when hcpcs_desc_1 like '%triglycerides%' then 'LAB/DIAGNOSTIC'
        else 'OTHER'
    end as TREATMENT
from
    members inner join inpatient_claims ic on members.member_id=ic.member_id
where
    members.CLIENT_ID=3;
"""

claimsAnalysisDF=pd.read_sql(text(claimsAnalysisQuery),claimsEngine)
print(claimsAnalysisDF.shape)

## Pull outpatient claims

Same logic, but outpatient visits have no discharge date, so the service date (CLM_FROM_DT) is used instead.

In [ ]:
outQuery="""
select
    members.member_id,
    oc.clm_from_dt as treatment_date,
    oc.clm_pmt_amt as claim_amount,
    case when icd9_dgns_desc_1 like '%shoulder%' then 'SHOULDER'
        when icd9_dgns_desc_1 like '%rotator cuff%' then 'SHOULDER'
        when icd9_dgns_desc_1 like '%knee%' then 'KNEE'
        when icd9_dgns_desc_1 like '%lower leg%' then 'KNEE'
        when icd9_dgns_desc_1 like '%pelvic region and thigh%' then 'HIP'
        when icd9_dgns_desc_1 like '%hip%' then 'HIP'
        when icd9_dgns_desc_1 like '%back%' then 'BACK'
        when icd9_dgns_desc_1 like '%lumbar%' then 'BACK'
        when icd9_dgns_desc_1 like '%lumbago%' then 'BACK'
        when icd9_dgns_desc_1 like '%intervertebral disc%' then 'BACK'
        else 'OTHER'
    end as CNDTN,
    case when hcpcs_desc_1 like '%arthroplasty%' then 'SURGERY'
        when hcpcs_desc_1 like '%arthrodesis%' then 'SURGERY'
        when hcpcs_desc_1 like '%laminotomy%' then 'SURGERY'
        when hcpcs_desc_1 like '%surgery%' then 'SURGERY'
        when hcpcs_desc_1 like '%therapeutic exercise%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%neuromuscular reeducation%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%manual therapy%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%therapy%' then 'PHYSICAL THERAPY'
        when hcpcs_desc_1 like '%injection%' then 'MEDICATION'
        when hcpcs_desc_1 like '%arthrocentesis%' then 'MEDICATION'
        when hcpcs_desc_1 like '%vaccine%' then 'MEDICATION'
        when hcpcs_desc_1 like '%radiologic%' then 'IMAGING'
        when hcpcs_desc_1 like '%mri%' then 'IMAGING'
        when hcpcs_desc_1 like '%office or other outpatient visit%' then 'OFFICE VISIT'
        when hcpcs_desc_1 like '%preventive medicine%' then 'OFFICE VISIT'
        when hcpcs_desc_1 like '%electrocardiogram%' then 'LAB/DIAGNOSTIC'
        when hcpcs_desc_1 like '%metabolic panel%' then 'LAB/DIAGNOSTIC'
        when hcpcs_desc_1 like '%triglycerides%' then 'LAB/DIAGNOSTIC'
        else 'OTHER'
    end as TREATMENT
from
    members inner join outpatient_claims oc on members.member_id=oc.member_id
where
    members.CLIENT_ID=3;
"""

outDF = pd.read_sql(text(outQuery), claimsEngine)
claimsAnalysisDF = pd.concat([claimsAnalysisDF, outDF], ignore_index=True)
print(claimsAnalysisDF.shape)
print(claimsAnalysisDF['CNDTN'].value_counts(dropna=False))
print(claimsAnalysisDF['TREATMENT'].value_counts(dropna=False))

## Join calls to claims through the crosswalk

In [ ]:
from pandasql import sqldf

query="""
select
    crosswalk.MEMBER_ID,
    callAnalysisDF.CALL_MEMBER_ID,
    callAnalysisDF.call_date as CALL_DATE,
    callAnalysisDF.pre_intent as PRE_INTENT,
    callAnalysisDF.post_intent as POST_INTENT,
    callAnalysisDF.condition as CONDITION_CALLS,
    claimsAnalysisDF.treatment_date as TREATMENT_DATE,
    claimsAnalysisDF.TREATMENT,
    claimsAnalysisDF.claim_amount as CLAIM_AMOUNT,
    claimsAnalysisDF.CNDTN as CONDITION_CLAIMS
from
    callAnalysisDF inner join
    crosswalk on callAnalysisDF.CALL_MEMBER_ID=crosswalk.CALL_MEMBER_ID inner join
    claimsAnalysisDF on claimsAnalysisDF.member_id=crosswalk.MEMBER_ID
"""

behaviorAnalysisDF=sqldf(query, globals())
print(behaviorAnalysisDF.shape)

## Behavior analysis dataset

In [ ]:
print(behaviorAnalysisDF)

## Pull every nurse call with pre- and post-call intent

847 calls for 735 members, some members had follow-up calls.

In [ ]:
q = "SELECT nurse_calls.* from dbo.nurse_calls inner join dbo.members on nurse_calls.CALL_MEMBER_ID=members.CALL_MEMBER_ID where members.CLIENT_ID=103"
callData = pd.read_sql(text(q), callEngine)
print(callData.shape)
callData.head()

## Rank treatments by intensity

Surgery 3, Physical Therapy 2, Medication 1, Self-Care 0. A negative shift means the member moved to a less intensive, lower-cost treatment during the call.

In [ ]:
aLittleDictionary = {'Surgery':3, 'Physical Therapy':2, 'Medication':1, 'Self-Care':0}

callData['PRE_RANK']  = callData['PRE_INTENT'].map(aLittleDictionary)
callData['POST_RANK'] = callData['POST_INTENT'].map(aLittleDictionary)
callData['SHIFT'] = callData['POST_RANK'] - callData['PRE_RANK']

print("Unmapped PRE:", callData['PRE_RANK'].isna().sum())
print("Unmapped POST:", callData['POST_RANK'].isna().sum())
print("\nDownshifted (cheaper after call):", (callData['SHIFT'] < 0).sum())
print("No change:", (callData['SHIFT'] == 0).sum())
print("Upshifted:", (callData['SHIFT'] > 0).sum())

## Intent before vs. after the call

Rows are what the member intended going in, columns what they intended coming out. The diagonal is no change; everything below it is a downshift.

In [ ]:
order = ['Surgery','Physical Therapy','Medication','Self-Care']
matrix = pd.crosstab(callData['PRE_INTENT'], callData['POST_INTENT']).reindex(index=order, columns=order, fill_value=0)
print(matrix)

## Benchmark the cost of each treatment

Average claim amount per treatment category, taken from the actual inpatient and outpatient claims. Self-Care is $0 because it generates no claim.

In [ ]:
costs = claimsAnalysisDF.groupby('TREATMENT')['claim_amount'].mean().round(2)
print(costs)

costMap = {
    'Surgery':          costs.get('SURGERY', 0),
    'Physical Therapy': costs.get('PHYSICAL THERAPY', 0),
    'Medication':       costs.get('MEDICATION', 0),
    'Self-Care':        0.0,
}
print(costMap)

## Estimate savings per call

Savings = average cost of the intended treatment − average cost of the treatment chosen after the call. Negative values are floored at zero so upshifts don't offset the total.

In [ ]:
callData['PRE_COST']  = callData['PRE_INTENT'].map(costMap)
callData['POST_COST'] = callData['POST_INTENT'].map(costMap)
callData['SAVINGS']   = (callData['PRE_COST'] - callData['POST_COST']).clip(lower=0)

savings = callData[callData['SHIFT'] < 0]
print("Calls with savings:", len(savings))
print("Total estimated savings: $", round(savings['SAVINGS'].sum(), 2))
print("Average per call: $", round(savings['SAVINGS'].mean(), 2))

## Savings by type of shift

In [ ]:
summary = (savings.groupby(['PRE_INTENT','POST_INTENT'])
           .agg(CALLS=('SAVINGS','size'), AVG_SAVED=('SAVINGS','mean'), TOTAL_SAVED=('SAVINGS','sum'))
           .round(2).sort_values('TOTAL_SAVED', ascending=False))
print(summary)

## Results

In [ ]:
print("=== OCC PROGRAM COST SAVINGS ===")
print(f"Total calls analyzed:        {len(callData)}")
print(f"Calls shifted to lower cost: {len(savings)}")
print(f"Total estimated savings:     ${savings['SAVINGS'].sum():,.2f}")
print(f"Average savings per call:    ${savings['SAVINGS'].mean():,.2f}")
print()
print(summary)

## Check for double-counting

Some members had more than one call, so savings are also totaled at the member level to confirm no decision is counted twice.

In [ ]:
perMember = savings.groupby('CALL_MEMBER_ID')['SAVINGS'].max()
print("Unique members with savings:", len(perMember))
print("Member-level total: $", round(perMember.sum(), 2))